# AMD/NVIDIA 原始 SEC 前瞻收入增长数值口径首轮

只读 126 个已封存来源事件，优先绝对金额除同份公告实际收入，缺省才用明确环比百分比；无法解释的格式保持未知，不读市场收益。

In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_amd_nvda_guidance_breadth_source_20261023';plan=json.loads((card/'plan.json').read_text());parent=root/plan['parent']['source_card'];raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(parent/'quarter_event_adjudication_result.json')==plan['parent']['quarter_event_result_sha256'] and sha(parent/'item202_inventory_result.json')==plan['parent']['item202_inventory_result_sha256'] and sha(parent/'independent_source_review.json')==plan['parent']['independent_source_review_sha256'] and not (card/'numeric_attempt1_result.json').exists()
source=json.loads((parent/'quarter_event_adjudication_result.json').read_text());assert len(source['selected_events'])==126 and source['parent_NAS_archive_sha256']==plan['parent']['original_full_submission_NAS_archive_sha256']
word={'zero':0,'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,'eight':8,'nine':9,'ten':10,'eleven':11,'twelve':12,'fifteen':15,'twenty':20}
def amount_usd_m(number,unit):
 val=float(number.replace(',',''));return val*(1000 if unit.lower().startswith('billion') else 1)
def value(token):
 token=token.lower().strip();return float(token.replace(',','')) if re.fullmatch(r'[\d,.]+',token) else float(word[token]) if token in word else None
def release_text(body):
 parts=[]
 for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',body,re.S|re.I):
  typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
  if typ and content and typ.group(1).strip().upper() in (b'EX-99.1',b'EX-95.1'):
   parts.append(re.sub(r'\s+',' ',BeautifulSoup(content.group(1).decode('utf8','replace'),'html.parser').get_text(' ',strip=True)))
 return ' '.join(parts)
rows=[]
for event in source['selected_events']:
 acc=event['accession'];p=raw/f'item202_{acc}.txt';assert sha(p)==event['raw_sha256'];body=p.read_bytes();text=release_text(body);issuer=event['issuer']
 if event['revenue_outlook']=='qualitative_unknown':
  rows.append({'issuer':issuer,'filing_date':event['filing_date'],'accession':acc,'raw_sha256':event['raw_sha256'],'status':'qualitative_unknown','growth':None,'format':None,'guidance_excerpt':None});continue
 candidates=[]
 for marker in re.finditer(r'\b(?:current\s+)?outlook\b',text,re.I):
  segment=text[marker.end():marker.end()+1800]
  found=re.search(r'(?:Revenue\s+is\s+expected|AMD\s+expects\s+revenue|expects\s+revenue\s+to|Revenue\s+to\s+be)',segment,re.I)
  if found:candidates.append((found.start(),segment[found.start():found.start()+400]))
 clause=min(candidates,key=lambda z:z[0])[1] if candidates else ''
 status='unresolved';format_=None;growth=None;reported=None;forecast=None;reason='no_supported_numeric_forward_clause'
 # Absolute USD ranges and central values take precedence over stated sequential percentages.
 between=re.search(r'between\s+\$?\s*([\d,.]+)\s*(million|billion)?\s+and\s+\$?\s*([\d,.]+)\s*(million|billion)',clause,re.I)
 absolute=re.search(r'\$\s*([\d,.]+)\s*(million|billion)',clause,re.I)
 if between:
  unit1=between.group(2) or between.group(4);forecast=(amount_usd_m(between.group(1),unit1)+amount_usd_m(between.group(3),between.group(4)))/2;format_='absolute_range'
 elif absolute:
  forecast=amount_usd_m(absolute.group(1),absolute.group(2));format_='absolute_central'
 if forecast is not None:
  pre=text[:max(0,text.find(clause[:35]))] if clause else text
  patterns=[r'(?:today\s+(?:reported|announced)|quarter\s+revenue\s+was|revenue\s+was).{0,120}?revenue\s+(?:of|was)\s*\$\s*([\d,.]+)\s*(million|billion)',r'\brevenue\s+(?:of|was)\s*\$\s*([\d,.]+)\s*(million|billion)']
  for pattern in patterns:
   m=re.search(pattern,pre,re.I)
   if m:reported=amount_usd_m(m.group(1),m.group(2));break
  if reported and reported>0:
   growth=forecast/reported-1;status='numeric';reason='absolute_guidance_over_same_release_current_revenue'
  else:reason='current_revenue_not_parseable'
 if growth is None:
  pct=re.search(r'(?:\bup\b|\bincrease\b|\bdecrease\b|\bdown\b)\s+([\d,.]+|[a-z]+)(?:\s+to\s+([\d,.]+|[a-z]+))?\s*(?:percent|%)',clause,re.I)
  if pct:
   lo=value(pct.group(1));hi=value(pct.group(2)) if pct.group(2) else lo
   if lo is not None and hi is not None:
    sign=-1 if re.search(r'\b(?:decrease|down)\s+'+re.escape(pct.group(1)),pct.group(0),re.I) else 1
    growth=sign*(lo+hi)/200;status='numeric';format_='direct_sequential_percent';reason='direct_next_quarter_sequential_percent'
  elif re.search(r'\bflat\b.{0,30}(?:plus\s+or\s+minus|\+/-)\s+(?:two|2)\s*(?:percent|%)',clause,re.I):
   growth=0.0;status='numeric';format_='direct_flat_plus_minus_percent';reason='explicit_numeric_symmetric_flat_guidance'
 if growth is not None and not (-1<growth<3):status='unresolved';reason='growth_out_of_semantic_bounds';growth=None
 rows.append({'issuer':issuer,'filing_date':event['filing_date'],'accession':acc,'raw_sha256':event['raw_sha256'],'status':status,'format':format_,'growth':growth,'reported_current_USD_million':reported,'forecast_next_mid_USD_million':forecast,'guidance_excerpt':clause[:270],'reason':reason})
assert len(rows)==126
by={issuer:{'numeric':sum(x['issuer']==issuer and x['status']=='numeric' for x in rows),'unknown':sum(x['issuer']==issuer and x['status']!='numeric' for x in rows)} for issuer in ('AMD','NVDA')}
result={'status':'first_numeric_harmonization_attempt_source_only_unresolved_preserved','plan_sha256':sha(card/'plan.json'),'parent_event_result_sha256':plan['parent']['quarter_event_result_sha256'],'rows':rows,'by_issuer':by,'unresolved':[{'issuer':x['issuer'],'filing_date':x['filing_date'],'accession':x['accession'],'reason':x.get('reason'),'guidance_excerpt':x.get('guidance_excerpt')} for x in rows if x['status']!='numeric'],'new_GETs':0,'Qlib_calls':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'ready_for_Qlib_map':False};(card/'numeric_attempt1_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'by_issuer':by,'unresolved_count':len(result['unresolved']),'formats':{f:sum(x['format']==f for x in rows) for f in sorted({x['format'] for x in rows if x['format']})}})


{'status': 'first_numeric_harmonization_attempt_source_only_unresolved_preserved', 'by_issuer': {'AMD': {'numeric': 56, 'unknown': 7}, 'NVDA': {'numeric': 55, 'unknown': 8}}, 'unresolved_count': 15, 'formats': {'absolute_central': 93, 'absolute_range': 5, 'direct_sequential_percent': 23}}


## 中文结论

首轮源数值解析发现全年/分部收入与费用误入，结果保留不入模。 见 `conclusion.md`。